# 🧠 คำอธิบายและตัวอย่างการปฏิบัติการการถดถอยโลจิสติก (Logistic Regression)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **การถดถอยโลจิสติก (Logistic Regression)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลองสำหรับจำแนกประเภทแบบสองกลุ่ม (Binary Classification) อ้างอิงตามกรณีศึกษาการแยกประเภทเกจวัด (`analog-gauge` (เกจเข็ม/อนาล็อก) เทียบกับ `digital-gauge` (เกจดิจิทัล/หน้าจอ))
2. ฝึกสอนแบบจำลองจำแนกประเภท **Logistic Regression** โดยใช้ไลบรารี `scikit-learn`
3. พัฒนาแบบจำลอง **Logistic Regression จากศูนย์ (from scratch)** โดยใช้กระบวนการ **Gradient Descent** และ **ฟังก์ชันซิกมอยด์ (Sigmoid Function)**
4. พล็อตกราฟการลดลงของค่าความสูญเสียสะสม (Loss) ตลอดรอบการฝึกสอน
5. จำลองภาพแสดง **ขอบเขตการตัดสินใจ (Decision Boundary)** ที่แบ่งข้อมูลออกเป็นสองคลาส
6. เชื่อมโยงแนวคิดนี้เข้ากับหัวจำแนกประเภท (Classification Heads) ในสถาปัตยกรรม Deep Learning (เช่น YOLO)

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. การสร้างข้อมูลตามกรณีศึกษา (Case Study Data Generation)

เราจะสุ่มสร้างข้อมูลเกจวัด 100 ตัวอย่าง ประกอบไปด้วย 2 คุณลักษณะ:
1.  `aspect_ratio` อัตราส่วนภาพ (เกจวัดแบบอนาล็อกมักมีทรงกลมหรือสี่เหลี่ยมจัตุรัส: ค่าประมาณ ~1.0 ขณะที่เกจวัดดิจิทัลที่เป็นหน้าจอมักมีความกว้างมากกว่า: ค่าประมาณ ~1.7)
2.  `text_density` ความหนาแน่นของตัวอักษร (หน้าจอดิจิทัลจะมีการแสดงผลอักษรและตัวเลขอัดแน่นอยู่มากกว่ามาก ส่วนเกจวัดอนาล็อกจะมีเพียงขีดและเข็มเป็นหลัก)

สังเคราะห์ข้อมูล:
*   Class 1 (`analog-gauge`): อัตราส่วนภาพเฉลี่ยอยู่ที่ 1.0, ความหนาแน่นตัวอักษรเฉลี่ยอยู่ที่ 2.0
*   Class 0 (`digital-gauge`): อัตราส่วนภาพเฉลี่ยอยู่ที่ 1.7, ความหนาแน่นตัวอักษรเฉลี่ยอยู่ที่ 5.5

In [ ]:
m = 100 # จำนวนตัวอย่างข้อมูล

# สร้างข้อมูลเกจวัดอนาล็อก (คลาส 1)
X_analog = np.random.randn(m // 2, 2) * 0.2 + np.array([1.0, 2.0])
y_analog = np.ones(m // 2)

# สร้างข้อมูลเกจวัดดิจิทัล (คลาส 0)
X_digital = np.random.randn(m // 2, 2) * 0.2 + np.array([1.7, 5.5])
y_digital = np.zeros(m // 2)

# รวมชุดข้อมูล
X = np.vstack((X_analog, X_digital))
y = np.concatenate((y_analog, y_digital))

# พล็อตกราฟแสดงการกระจายตัวของชุดข้อมูล
plt.figure(figsize=(8, 5))
plt.scatter(X_analog[:, 0], X_analog[:, 1], color='blue', label='Class 1: Analog Gauge', alpha=0.7)
plt.scatter(X_digital[:, 0], X_digital[:, 1], color='red', label='Class 0: Digital Gauge', alpha=0.7)
plt.xlabel('Aspect Ratio')
plt.ylabel('Text Density')
plt.title('Gauge Classification Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 2. การทำ Logistic Regression ด้วย Scikit-Learn

ลองนำตัวจำแนกประเภทแบบโลจิสติกนี้มาฟิตเข้ากับชุดข้อมูล พร้อมตรวจดูค่าสัมประสิทธิ์ที่โมเดลเรียนรู้ได้

In [ ]:
# ฝึกสอนแบบจำลองด้วย Scikit-Learn
model = LogisticRegression()
model.fit(X, y)

# ดึงค่าน้ำหนักสัมประสิทธิ์และอคติ
sklearn_w1, sklearn_w2 = model.coef_[0]
sklearn_b = model.intercept_[0]

print(f"Skikit-Learn parameters:")
print(f"w_1 (Aspect Ratio Coefficient): {sklearn_w1:.4f}")
print(f"w_2 (Text Density Coefficient) : {sklearn_w2:.4f}")
print(f"Bias (Intercept)               : {sklearn_b:.4f}")

# ประเมินผลค่าความถูกต้อง (Accuracy)
y_pred_sklearn = model.predict(X)
print(f"Training Accuracy: {accuracy_score(y, y_pred_sklearn) * 100:.2f}%")

## 3. การสร้าง Logistic Regression จากศูนย์ด้วย Gradient Descent (Logistic Regression from Scratch)

ทบทวนสมการคณิตศาสตร์หลัก:
ฟังก์ชันซิกมอยด์ (Sigmoid function):
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

สมมติฐานและการทำนายผล (Hypothesis prediction):
$$\hat{y}^{(i)} = \sigma(\mathbf{w}^T \mathbf{x}^{(i)} + b)$$

ฟังก์ชันความสูญเสียประเภทไบนารีครอสเอนโทรปี (Binary Cross-Entropy Loss):
$$J(w, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

เกรเดียนต์เพื่อใช้ปรับค่า (Gradients):
$$\frac{\partial J}{\partial \mathbf{w}} = \frac{1}{m} \mathbf{X}^T (\hat{\mathbf{y}} - \mathbf{y})$$
$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

มาเริ่มเขียนโค้ดใช้งานกันครับ

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_loss(y, y_pred):
    # บวกค่า epsilon เล็กน้อยเพื่อป้องกันข้อผิดพลาดทางคณิตศาสตร์จาก log(0)
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y * np.log(y_pred) + (1 - y) * np.log(1 - y_pred))

# ไฮเปอร์พารามิเตอร์
learning_rate = 0.5
epochs = 500
m, n = X.shape

# กำหนดค่าเริ่มต้นให้น้ำหนักและอคติเป็น 0
w_custom = np.zeros(n)
b_custom = 0.0

loss_history = []

for epoch in range(epochs):
    # 1. การส่งผ่านข้อมูลไปข้างหน้า (Forward Pass เพื่อหาผลทำนาย)
    z = X @ w_custom + b_custom
    y_pred = sigmoid(z)
    
    # 2. คำนวณค่า Loss
    loss = compute_loss(y, y_pred)
    loss_history.append(loss)
    
    # 3. คำนวณค่าเกรเดียนต์ย้อนกลับ (Backward Pass)
    dw = (1 / m) * (X.T @ (y_pred - y))
    db = np.mean(y_pred - y)
    
    # 4. ปรับปรุงน้ำหนักและอคติ
    w_custom -= learning_rate * dw
    b_custom -= learning_rate * db
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | w: {w_custom} | b: {b_custom:.4f}")

print("\nFinal Learned Parameters (Custom Scratch):")
print(f"w: {w_custom} | b: {b_custom:.4f}")

มาพล็อตกราฟเส้นการลดลงของฟังก์ชันความสูญเสียไบนารีครอสเอนโทรปี (BCE Loss) กันครับ

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history, color='orange', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('BCE Loss Reduction Curve')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 4. การแสดงขอบเขตการตัดสินใจ (Decision Boundary Visualization)

ขอบเขตการตัดสินใจ (Decision Boundary) คือแนวขอบเขตที่โมเดลให้ค่าความน่าจะเป็นทำนายก้ำกึ่งที่ 0.5 พอดี ซึ่งจะเกิดขึ้นเมื่อสมการเส้นตรงของแบบจำลองมีค่า $z = 0$:
$$w_1 x_1 + w_2 x_2 + b = 0$$

เมื่อแก้สมการหาแกน $x_2$ (Text Density) เพื่อใช้วาดรูป:
$$x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

เรามาลองพล็อตเส้นแบ่งเขตนี้ทับจุดข้อมูลจริงกันครับ!

In [ ]:
plt.figure(figsize=(10, 6))

# พล็อตจุดข้อมูลดิบ
plt.scatter(X_analog[:, 0], X_analog[:, 1], color='blue', label='Class 1: Analog Gauge', alpha=0.6)
plt.scatter(X_digital[:, 0], X_digital[:, 1], color='red', label='Class 0: Digital Gauge', alpha=0.6)

# สร้างค่า X สำหรับลากเส้นแบ่งเขตการตัดสินใจ
x_boundary = np.linspace(0.6, 2.0, 100)

# ขอบเขตการตัดสินใจของ Scikit-Learn
y_boundary_sklearn = -(sklearn_w1 / sklearn_w2) * x_boundary - (sklearn_b / sklearn_w2)
plt.plot(x_boundary, y_boundary_sklearn, color='green', linewidth=2, label='Sklearn Decision Boundary')

# ขอบเขตการตัดสินใจของแบบจำลองที่เราทำขึ้นเองจากศูนย์
y_boundary_custom = -(w_custom[0] / w_custom[1]) * x_boundary - (b_custom / w_custom[1])
plt.plot(x_boundary, y_boundary_custom, color='purple', linestyle='--', linewidth=2, label='Scratch Decision Boundary')

plt.xlabel('Aspect Ratio')
plt.ylabel('Text Density')
plt.title('Logistic Regression Decision Boundary Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(1.0, 6.5)
plt.show()

## 💡 ความเชื่อมโยงสู่ Deep Learning และ YOLO
*   **ชั้นสำหรับการจำแนกประเภท (ฟังก์ชันการเปิดใช้งานแบบ Sigmoid):** ในงานจำแนกประเภทวัตถุที่มีได้หลายคลาสพร้อมกัน (Multi-label classification) ของแบบจำลองชั้นนำอย่าง YOLO คะแนนทำนายคลาสของกล่องวัตถุแต่ละกล่องจะถูกคำนวณผ่านฟังก์ชันเปิดใช้งานแบบ **Sigmoid** โดย YOLO จะใช้ **Binary Cross-Entropy Loss** ในส่วนของการจำแนกประเภทวัตถุ ซึ่งเปรียบเหมือนการฝึกสอนหัวทำนายแบบการถดถอยโลจิสติก (Logistic Regression Heads) แยกอิสระต่อกันทีละตัวสำหรับแต่ละคลาส เพื่อตอบคำถามว่าในกรอบนั้นๆ มีวัตถุประเภะนั้นปรากฏอยู่จริงหรือไม่ (เช่น มี `lever-valve` หรือไม่, มี `control-valve` หรือไม่)